# TouristGuide Agent — Deliverable 1: Working Prototype

This notebook demonstrates the complete agent loop across **6 test scenarios**:
1. Multi-tool request (all 4 tools)
2. Clarification scenario (missing budget)
3. Tool failure (invalid city)
4. Budget too low
5. Multi-day trip with currency conversion
6. Unlisted city (Overpass fallback)


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv()
from core.agent import plan_trip
print('Agent loaded.')

---
## Scenario 1 — Multi-Tool Request
All four tools (weather, attractions, currency, itinerary) are exercised.

> "Plan a one-day trip in Kampala for someone with a budget of 50 USD, show costs in UGX."

In [ ]:
result = plan_trip({
    'city': 'Kampala',
    'num_days': 1,
    'budget': 50,
    'from_currency': 'USD',
    'to_currency': 'UGX'
})

print('Status         :', result['status'])
print('Tools used     :', list(result.get('tool_outputs', {}).keys()))
print('Errors         :', [e['tool'] for e in result.get('errors', [])])
print()
print('--- ITINERARY ---')
print(result.get('itinerary', ''))
print()
print('--- WEATHER ---')
w = result.get('tool_outputs', {}).get('weather', {}).get('weather', {})
print(f"{w.get('condition')}, {w.get('temperature')}°C, humidity {w.get('humidity')}%, wind {w.get('wind_kph')} km/h")
print()
print('--- CURRENCY ---')
c = result.get('tool_outputs', {}).get('currency', {})
print(f"{c.get('original_amount'):,} {c.get('from_currency')} = {c.get('converted_amount'):,} {c.get('to_currency')} (rate: {c.get('rate')})")


---
## Scenario 2 — Clarification Scenario
User provides city but omits budget. Agent should ask for the missing field.

> "Plan me a trip in Kigali."

In [ ]:
result = plan_trip({'city': 'Kigali'})

print('Status         :', result['status'])
print('Missing fields :', result.get('missing_fields', []))
print()
print('Agent asks     :', result.get('message', ''))


---
## Scenario 3 — Tool Failure (Invalid City)
City name is nonsensical. Weather and attractions tools will fail; agent should recover gracefully.

> "Plan a trip to Xyzabc123 with a budget of 100 USD."

In [ ]:
result = plan_trip({
    'city': 'Xyzabc123',
    'num_days': 1,
    'budget': 100,
    'from_currency': 'USD',
    'to_currency': 'USD'
})

print('Status         :', result['status'])
print('Tools succeeded:', list(result.get('tool_outputs', {}).keys()))
print('Tools failed   :', [e['tool'] for e in result.get('errors', [])])
print()
print('Agent response :', result.get('itinerary', ''))


---
## Scenario 4 — Budget Too Low
User's budget is far below the minimum daily cost for the destination.

> "Plan a 3-day trip in Paris with a budget of 10 USD."

In [ ]:
result = plan_trip({
    'city': 'Paris',
    'num_days': 3,
    'budget': 10,
    'from_currency': 'USD',
    'to_currency': 'EUR'
})

print('Status:', result['status'])
print()
print(result.get('error', ''))


---
## Scenario 5 — Multi-Day Trip with Currency Conversion
Two-day trip with JPY budget converted to USD. Tests attraction rotation and weather advisory.

> "Plan a 2-day trip in Tokyo with a budget of 50,000 JPY, show costs in USD."

In [ ]:
result = plan_trip({
    'city': 'Tokyo',
    'num_days': 2,
    'budget': 50000,
    'from_currency': 'JPY',
    'to_currency': 'USD'
})

print('Status         :', result['status'])
print('Tools used     :', list(result.get('tool_outputs', {}).keys()))
print()
print('--- ITINERARY ---')
print(result.get('itinerary', ''))
print()
print('--- CURRENCY ---')
c = result.get('tool_outputs', {}).get('currency', {})
print(f"{c.get('original_amount'):,} {c.get('from_currency')} = {c.get('converted_amount'):,} {c.get('to_currency')} (rate: {c.get('rate')})")


---
## Scenario 6 — Unlisted City (API/Overpass Fallback)
City not in the 54-city local attractions DB. System falls back to OpenStreetMap Overpass.

> "Plan a day trip in Lomé with a budget of 100,000 XOF."

In [ ]:
result = plan_trip({
    'city': 'Lomé',
    'num_days': 1,
    'budget': 100000,
    'from_currency': 'XOF',
    'to_currency': 'USD'
})

print('Status         :', result['status'])
print('Tools succeeded:', list(result.get('tool_outputs', {}).keys()))
print('Tools failed   :', [e['tool'] for e in result.get('errors', [])])
print()
attractions = result.get('tool_outputs', {}).get('attractions', {})
if attractions:
    print('Attractions source:', attractions.get('source'))
    for a in attractions.get('attractions', []):
        print(' -', a)
print()
print('Agent response :', result.get('itinerary', ''))


---
## Summary

| Scenario | Status | Tools used |
|---|---|---|
| 1. Multi-tool request (Kampala) | success | weather, attractions, currency, itinerary |
| 2. Clarification (Kigali, no budget) | needs_clarification | — |
| 3. Tool failure (invalid city) | partial_success | currency only |
| 4. Budget too low (Paris, $10) | budget_too_low | — |
| 5. Multi-day trip (Tokyo, JPY→USD) | success | weather, attractions, currency, itinerary |
| 6. Unlisted city fallback (Lomé) | partial_success / success | weather, currency (+ attractions if Overpass responds) |